# Kaggriculture · Terminal cash feature ablation
## Notebook 09 | Does a better inventory representation improve the terminal result?

Notebook 08 found an activated post-action inventory mechanism. This notebook keeps that intervention unchanged and evaluates **paired, reactive final-day continuations**. No new initial-state games, model search, installations or cloud modifications are performed.

**Research status:** open. Two development seeds, one related opponent, three primary comparisons and four negative controls. Results are not leaderboard ratings. The 12 new terminal-exposure candidates are logged only, not used to change actions.

## Reviewed mechanics correction
The previous run stopped before its first research branch. Its fixture incorrectly
assumed WHEAT at supply 1,000,000 costs one coin. The pinned curve quotes 13;
two units sell for 26, not 2. The revised gate calculates each unit's quote before
the simulation and verifies absolute cash, residual stock and market inventory.
The feature intervention, paired sources, actors and stop limits are unchanged.
The updater preserves failure diagnostics and authorizes one code-bound restart;
it does not authorize repeated retries after a different failure.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Markdown, FileLink

BASE = Path.cwd().resolve()
if not (BASE / 'run_continuation.py').is_file():
    BASE = Path.home() / 'kaggriculture_terminal_ablation'
assert (BASE / 'run_continuation.py').is_file(), 'Open this notebook from the extracted package.'
runtime = json.loads((Path.home() / 'kaggriculture_manual_resume/state/runtime.json').read_text())
assert Path(sys.executable).absolute() == Path(runtime['executable']).absolute(), 'Choose Kaggriculture Manual (verified source).'
assert sys.version_info[:2] == (3, 12), 'Use the existing verified Python 3.12 runtime.'
sys.path.insert(0, str(BASE))
OUT = BASE / 'outputs'
MODE = 'AWS_REACTIVE_FINAL_DAY_CONTINUATIONS'
print('Kernel:', sys.executable)
print('Mode:', MODE)
print('No preceding notebook experiments will be rerun.')

Kernel: /home/sagemaker-user/projects/kaggriculture/.venv/bin/python
Mode: AWS_REACTIVE_FINAL_DAY_CONTINUATIONS
No preceding notebook experiments will be rerun.


### 1 · Evidence entering this milestone
These are measured notebook-08 results, not new experiment outcomes. Most positive changes occurred before the final callback, so they may be timing effects.

In [2]:
review = json.loads((BASE / 'reference/input_review.json').read_text())
display(pd.DataFrame([
    ('Previous status', review['status']),
    ('Saved observations / source episodes', f"{review['observations']} / {review['episodes']}"),
    ('Market-action changes', review['market_changes']),
    ('Positive / negative isolated cash effects', f"{review['positive_one_step_states']} / {review['negative_one_step_states']}"),
    ('Largest isolated cash difference', review['largest_one_step_gain_coins']),
    ('Positive final-callback difference', review['positive_final_callback_rows'][0]['coins_delta']),
], columns=['Evidence', 'Measured result']))

,Evidence,Measured result
0,Previous status,POST_ACTION_CASH_AUDIT_PASSED
1,Saved observations / source episodes,161 / 7
2,Market-action changes,11
3,Positive / negative isolated cash effects,6 / 0
4,Largest isolated cash difference,1440
5,Positive final-callback difference,42


### 2 · Bounded execution
The worker stops at 180 seconds, saves each completed branch, and refuses unchanged retries after failures. A harmful completed primary pair stops expansion. The live opponent sees only its own evolving legal observation.

In [3]:
import os
import selectors
import signal
import subprocess
import time

def run_bounded_notebook_stage():
    command = [sys.executable, str(BASE / 'run_continuation.py'), 'run', '--seconds', '180']
    proc = subprocess.Popen(command, cwd=BASE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True,
        env={**os.environ, 'PYTHONUNBUFFERED':'1', 'PYTHONDONTWRITEBYTECODE':'1'})
    selector = selectors.DefaultSelector()
    selector.register(proc.stdout, selectors.EVENT_READ)
    started = time.monotonic()
    try:
        while proc.poll() is None:
            if time.monotonic() - started > 210:
                # run_continuation owns a separate worker group and an inner hard deadline.
                # Terminating its parent also invokes its worker cleanup handler.
                os.killpg(proc.pid, signal.SIGINT)
                try: proc.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    os.killpg(proc.pid, signal.SIGKILL); proc.wait()
                raise TimeoutError('Notebook emergency deadline. Bundle diagnostics; do not retry unchanged.')
            for key, _ in selector.select(timeout=1):
                line = key.fileobj.readline()
                if line: print(line.rstrip(), flush=True)
        for line in proc.stdout: print(line.rstrip(), flush=True)
        if proc.returncode:
            raise RuntimeError('Notebook 09 stopped. Save and run the bundle command from START_HERE.md.')
    finally:
        selector.close()
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGINT)
            try: proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL); proc.wait()

run_bounded_notebook_stage()

{"utc": "2026-09-11T23:58:01.063534+00:00", "stage": "REVIEWED_MECHANICS_RESTART_AUTHORIZED", "update_id": "nb09-price-aware-mechanics-1"}
......................................................................................
----------------------------------------------------------------------
Ran 86 tests in 4.874s

OK
{"utc": "2026-09-11T23:58:06.727437+00:00", "stage": "PREFLIGHT_PASSED", "verified_objects": 10}
{"utc": "2026-09-11T23:58:06.738317+00:00", "stage": "MECHANICS_FIXTURE", "fixture": "early_sale_catches_up", "initial_quote": 13, "expected_delta": 0, "observed_delta": 0.0, "passed": true}
{"utc": "2026-09-11T23:58:06.746801+00:00", "stage": "MECHANICS_FIXTURE", "fixture": "last_callback_stranding", "initial_quote": 13, "expected_delta": 26, "observed_delta": 26.0, "passed": true}
{"utc": "2026-09-11T23:58:06.746926+00:00", "stage": "MECHANICS_PASSED", "fixtures": 2, "interpreter_calls": 12}
{"utc": "2026-09-11T23:58:07.399217+00:00", "stage": "BRANCH_STARTED", "episode"

## Mechanics gate: expected versus observed
These are synthetic mechanics fixtures evaluated by the pinned interpreter, not
competitive scores. Both early-sale endpoints should be 3,026 coins; only the
aligned late-sale endpoint should be 3,026 (control remains 3,000).

In [4]:
import plotly.express as px
mechanics = pd.DataFrame(json.loads((OUT / 'mechanics.json').read_text()))
assert mechanics['passed'].all(), 'Mechanics gate did not pass.'
display(mechanics[['fixture', 'initial_quote', 'per_unit_quotes',
    'control_final_cash', 'aligned_final_cash', 'expected_delta', 'terminal_cash_delta']])
mechanics_plot = mechanics.melt(id_vars=['fixture'],
    value_vars=['expected_delta', 'terminal_cash_delta'],
    var_name='Measurement', value_name='Coins')
mechanics_fig = px.bar(mechanics_plot, x='fixture', y='Coins', color='Measurement',
    barmode='group', title='Mechanics fixtures — predicted and observed cash differences',
    labels={'fixture': 'Fixture'})
mechanics_fig.show()
mechanics_fig.write_html(OUT / 'mechanics_check.html', include_plotlyjs=True)


,fixture,initial_quote,per_unit_quotes,control_final_cash,aligned_final_cash,expected_delta,terminal_cash_delta
0,early_sale_catches_up,13,"[13, 13]",3026.0,3026.0,0,0.0
1,last_callback_stranding,13,"[13, 13]",3000.0,3026.0,26,26.0


### 3 · Endpoint decision
Completion means the bounded experiment returned a result, not that the intervention improved performance. Negative controls are separate from efficacy comparisons.

In [5]:
from run_continuation import verify_result
report = verify_result()
print('Status:', report['status'])
print('Decision:', report['decision'])
print('Completed primary / negative-control pairs:', report['primary_pairs'], '/', report['negative_control_pairs'])
print('New / represented suffix steps:', report['new_suffix_interpreter_calls'], '/', report['represented_suffix_interpreter_calls'])
print('New complete games from initial state:', report['new_complete_games_from_initial_state'])
print('Official metric effect measured:', report['official_metric_effect_measured'])
pairs = pd.read_csv(OUT / 'paired_results.csv')
display(pairs[['seed','seat','role','coins_control','coins_aligned','coins_delta','coin_margin_delta','local_match_score_delta','residual_product_units_delta']])

Status: TERMINAL_CASH_EXPERIMENT_COMPLETE
Decision: STOP_NEGATIVE_TERMINAL_EFFECT
Completed primary / negative-control pairs: 1 / 0
New / represented suffix steps: 46 / 46
New complete games from initial state: 0
Official metric effect measured: False


,seed,seat,role,coins_control,coins_aligned,coins_delta,coin_margin_delta,local_match_score_delta,residual_product_units_delta
0,1601,0,primary_intervention,44840.0,44837.0,-3.0,-3.0,0.0,0


### 4 · Previous one-step effects
This is the earlier evidence. Do not add these isolated branches into a season total.

In [6]:
from visualize_continuation import figures, export_dashboard
charts = figures(OUT, BASE / 'reference')
charts[0].show()

### 5 · Final cash on paired continuations
Both alternatives start from the same source state. The control must reproduce the recorded trajectory and final reward.

In [7]:
charts[1].show()

### 6 · Terminal effect and negative controls
Inspect cash, margin, match outcome and unsold inventory together. A 1,440-coin early sale is not necessarily a 1,440-coin endpoint gain.

In [8]:
charts[2].show()

### 7 · Does early cash become lasting cash?
Each point is a difference in current balances on two continuously evolving trajectories. Summing the points would double-count balances.

In [9]:
charts[3].show()

### 8 · Stability across observed seed/seat blocks
Blank cells are unobserved primary combinations—not zeros. Symmetric seats are not independent samples. No inferential confidence intervals are produced.

In [10]:
charts[4].show()
display(pd.read_csv(OUT / 'primary_by_seed.csv'))

,seed,evaluated_seats,primary_pairs,mean_coin_delta,min_coin_delta,max_coin_delta,mean_margin_delta,mean_local_match_delta,status
0,1601,0,1,-3.0,-3.0,-3.0,-3.0,0.0,descriptive_development_block_not_independent_...


### 9 · Complete-callback acceptance
These are actual candidate and opponent callback measurements on visited states. Same-state shadow controls are also gated. No hosted-Kaggle timing claim is made.

In [11]:
charts[5].show()
print('Candidate max ms:', report['candidate_callback_max_ms'])
print('Opponent max ms:', report['opponent_callback_max_ms'])
print('Same-state attribution:', report['same_state_attribution_all_passed'])

Candidate max ms: 38.053419993957505
Opponent max ms: 3.0471979989670217
Same-state attribution: True


### 10 · New feature family: liquidation exposure
These 12 candidates use current own observations only. Delivery is individually feasible or blocked by the terminal deadline; values are conditional single-seller scenarios. They are not used in this intervention or established feature importances.

In [12]:
charts[6].show()
dictionary = pd.read_csv(BASE / 'feature_dictionary.csv')
display(dictionary[['feature','definition','intervention_status']])

,feature,definition,intervention_status
0,terminal_context.callbacks_remaining,Number of actionable callbacks including the c...,Candidate logged for activation only; not used...
1,terminal_context.shed_product_units,Current own shed units that are market products.,Candidate logged for activation only; not used...
2,terminal_context.carried_product_units,Current product units carried by all own workers.,Candidate logged for activation only; not used...
3,terminal_context.individually_deliverable_carr...,Product cargo on workers with Manhattan distan...,Candidate logged for activation only; not used...
4,terminal_context.deadline_blocked_carried_units,Product cargo on workers that individually can...,Candidate logged for activation only; not used...
5,terminal_context.shed_liquidation_scenario_value,Exact single-seller sale value of current own ...,Candidate logged for activation only; not used...
6,terminal_context.deliverable_cargo_marginal_sc...,Incremental single-seller scenario value of in...,Candidate logged for activation only; not used...
7,terminal_context.deadline_blocked_cargo_margin...,Additional hypothetical single-seller value of...,Candidate logged for activation only; not used...
8,terminal_context.shed_room_before_actions,"100 minus all current shed units, including no...",Candidate logged for activation only; not used...
9,terminal_context.optimistic_delivery_overflow_...,"Max(0, all shed units + all independently deli...",Candidate logged for activation only; not used...


### 11 · Durable visual evidence

In [13]:
dashboard = OUT / 'terminal_ablation.html'
export_dashboard([mechanics_fig, *charts], dashboard, report['decision'], MODE)
display(FileLink(str(dashboard)))
print('Dashboard:', dashboard)

/home/sagemaker-user/kaggriculture_terminal_ablation/outputs/terminal_ablation.html

Dashboard: /home/sagemaker-user/kaggriculture_terminal_ablation/outputs/terminal_ablation.html


### 12 · Milestone decision and next research gate
Positive endpoint evidence remains provisional. Broader-workforce callback coverage, fresh groups, distinct opponents and a valid submission score remain outstanding. Stop nonproductive directions before expanding.

In [14]:
display(Markdown('**Decision:** ' + report['decision']))
for limitation in report['limitations']:
    print('•', limitation)
print('GitHub updated:', report['github_updated'])
print('SAVE NOTEBOOK (Ctrl+S), then run in a JupyterLab terminal:')
import shlex
print('cd ' + shlex.quote(str(BASE)))
print(shlex.quote(sys.executable) + ' run_continuation.py bundle')
print('Return: kaggriculture_terminal_ablation_results.zip')
print('Then stop the JupyterLab application; do not delete its space.')

**Decision:** STOP_NEGATIVE_TERMINAL_EFFECT

• Exploratory development study selected after notebook08; not preregistered before those observations.
• Seven preserved suffixes from two seed blocks and one related opponent; three coordinated primary pairs and four sequential negative controls.
• The missing coordinated seed1602/seat1 source episode stays missing. Negative controls are not additional efficacy trials.
• Seats can be symmetric; no turn-level inference, confidence intervals or claim of independent seven-game validation.
• A fresh live opponent responds according to its unchanged source policy; this is not retraining or a new independent opponent family.
• Frozen action callback still rejects more than three hands on either public farm.
• Twelve new terminal-context candidates are activation-only; conditional values ignore shared capacity and rival orders.
• Persistent local checkpoints plus a user-downloaded bundle, not S3 readback or an off-instance automatic backup.
• Local callback timings are not hosted Kaggle acc